# 09 · Señales técnicas y noticias

**Objetivo:** comprobar si lo que pasa después de una señal técnica de compra o de venta (notebook 05) cambia según las noticias de la empresa en los días anteriores. Se comparan las señales precedidas de noticias positivas, neutrales, negativas o de ninguna, medidas con M4, el mejor modelo de sentimiento (notebooks 07 y 08).

*Nada de lo que se muestra aquí es una recomendación de inversión.*

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Las señales del 05 (primer día de cada racha) y sus noticias previas: cuántas señales caen en cada grupo, sin mirar todavía ninguna rentabilidad | hecho |
| 2 | Rentabilidad a 5 y 10 sesiones menos SPY de cada grupo, frente a todas las señales (solo análisis técnico), y las comparaciones con su intervalo | hecho |

**Entradas:**
- `data/cache/senales.parquet` (notebook 05): las señales de compra y de venta de cada empresa y sesión
- `data/cache/eventos_noticias.parquet` (notebook 08): los eventos de noticias (empresa, sesión y sentimiento) de cada modelo
- `data/cache/precios_spy.parquet` (notebook 08): las sesiones de la bolsa y el mercado (SPY)
- `data/cache/precios_historicos.parquet` (notebook 02): los cierres, para la rentabilidad después de cada señal y la volatilidad de algunas empresas

**Salidas:**
- `data/cache/resultados_09_senales.parquet` y `resultados_09_comparaciones.parquet`: las tablas de la sección 2, para la app

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

from funciones import CARPETA_CACHE, SEMILLA, diferencia_medias_welch, media_con_intervalo

load_dotenv();

## 1. ¿Qué señales hay y qué noticias tenían antes?

**El diseño se fijó antes de mirar ninguna rentabilidad:**

- **Solo el primer día de cada racha de señales.** La ruptura de Donchian, por ejemplo, se repite varios días seguidos mientras el precio sigue subiendo; contar cada día sería contar la misma señal varias veces.
- **Solo empresas y fechas con noticias:** las 633 empresas con titulares de FNSPID, de octubre de 2016 a junio de 2020.
- **Las noticias previas son las de las 5 sesiones anteriores a la señal**, sin incluir la sesión de la señal: una noticia de ese día podría ser posterior al cierre en el que salta la señal, y eso sería mirar al futuro. **Como sensibilidad, 3 sesiones.** Las dos ventanas se declararon antes de ver resultados.
- **El grupo de la señal** sale del signo de la media del sentimiento de los eventos de noticias de la ventana (los del notebook 08, con M4): **positivas, neutrales, negativas**, o **sin noticias** si no hay ninguno.

Las señales, tal como las guardó el notebook 05: una fila por empresa y sesión, con la señal del día (`compra`, `venta` o vacía) y los mecanismos que la han disparado.

In [2]:
senales = pd.read_parquet(CARPETA_CACHE / "senales.parquet").sort_values(["ticker", "fecha"]).reset_index(drop=True)
print(f"Filas: {len(senales):,} · empresas: {senales['ticker'].nunique()} · del {senales['fecha'].min():%d %b %Y} al {senales['fecha'].max():%d %b %Y}")
print("Señales:", senales["senal"].value_counts().to_dict())
senales[senales["senal"].notna()].head(3)

Filas: 2,281,091 · empresas: 1000 · del 10 Oct 2016 al 09 Oct 2026
Señales: {'compra': 187837, 'venta': 129549}


,fecha,ticker,regimen,liquido,compra_ema,venta_ema,compra_rsi,venta_rsi,compra_donchian,venta_donchian,senal
61,2017-01-06,A,lateral,True,False,False,False,False,True,False,compra
62,2017-01-09,A,lateral,True,False,False,False,False,True,False,compra
64,2017-01-11,A,lateral,True,False,False,False,False,True,False,compra


### 1.1 El primer día de cada racha

Una racha son sesiones seguidas de una empresa con la misma señal. Se queda el primer día: el de una señal distinta de la de la sesión anterior.

In [3]:
senal_anterior = senales.groupby("ticker")["senal"].shift()
primeras = senales[senales["senal"].notna() & (senales["senal"] != senal_anterior)]
print(f"Días con señal: {senales['senal'].notna().sum():,} · primeros días de racha: {len(primeras):,}"
      f" ({len(primeras) / senales['senal'].notna().sum():.0%})")
primeras["senal"].value_counts()

Días con señal: 317,386 · primeros días de racha: 215,928 (68%)


senal
compra    124555
venta      91373
Name: count, dtype: int64

### 1.2 Las empresas y las fechas con noticias

Los eventos de noticias de M4 (notebook 08). Una señal entra si su empresa tiene noticias en FNSPID y si sus 5 sesiones previas caen dentro del periodo de las noticias.

In [4]:
eventos = pd.read_parquet(CARPETA_CACHE / "eventos_noticias.parquet")
eventos = eventos[eventos["modelo"] == "M4 FinBERT"].reset_index(drop=True)
sesiones = pd.DatetimeIndex(np.sort(pd.read_parquet(CARPETA_CACHE / "precios_spy.parquet", columns=["fecha"])["fecha"].unique()))
numero_de_sesion = pd.Series(np.arange(len(sesiones)), index=sesiones)

VENTANA, VENTANA_SENSIBILIDAD = 5, 3   # sesiones previas a la señal
primera_sesion_con_ventana = sesiones[numero_de_sesion[eventos["sesion"].min()] + VENTANA]
ultima_sesion_con_noticias = eventos["sesion"].max()

estudio = primeras[primeras["ticker"].isin(eventos["ticker"].unique())
                   & primeras["fecha"].between(primera_sesion_con_ventana, ultima_sesion_con_noticias)].copy()
estudio["sesion_numero"] = estudio["fecha"].map(numero_de_sesion)
print(f"Eventos de noticias de M4: {len(eventos):,} de {eventos['ticker'].nunique()} empresas")
print(f"Señales en el estudio: {len(estudio):,} de {estudio['ticker'].nunique()} empresas,"
      f" del {estudio['fecha'].min():%d %b %Y} al {estudio['fecha'].max():%d %b %Y} · {estudio['senal'].value_counts().to_dict()}")

Eventos de noticias de M4: 111,709 de 633 empresas
Señales en el estudio: 54,001 de 621 empresas, del 05 Jan 2017 al 11 Jun 2020 · {'compra': 32325, 'venta': 21676}


### 1.3 ¿Hay que quitar alguna empresa por tener el precio congelado?

En el notebook 05 aparecieron 7 empresas (TECH, MKTX, AES, ACA, CZR, LNTH y PEN) con un rango diario de solo el 0,25–0,5 % del precio, el patrón de una oferta de compra pendiente; con el precio anclado, una señal técnica no significa nada. Pero eso se midió en el último año (2025–2026), y aquí se estudia 2016–2020. Se mira el rango diario mediano (máximo menos mínimo, sobre el cierre) de cada empresa en el periodo del estudio:

In [5]:
precios = pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet", columns=["fecha", "ticker", "high", "low", "close"])
precios = precios[precios["fecha"].between(estudio["fecha"].min(), estudio["fecha"].max())]
rango_mediano = ((precios["high"] - precios["low"]) / precios["close"]).groupby(precios["ticker"]).median()

SIETE_DEL_05 = ["TECH", "MKTX", "AES", "ACA", "CZR", "LNTH", "PEN"]
print(f"Rango diario mediano de todas las empresas: {rango_mediano.median():.2%}")
print("Las 7 del notebook 05:", {ticker: f"{rango_mediano[ticker]:.2%}" for ticker in SIETE_DEL_05})
casi_quietas = rango_mediano[rango_mediano < 0.006]
print(f"Empresas con un rango por debajo del 0,6 %: {sorted(casi_quietas.index)}"
      f" · con señales en el estudio: {sorted(set(casi_quietas.index) & set(estudio['ticker']))}")

Rango diario mediano de todas las empresas: 1.90%
Las 7 del notebook 05: {'TECH': '1.96%', 'MKTX': '2.22%', 'AES': '1.85%', 'ACA': '3.18%', 'CZR': '3.30%', 'LNTH': '3.98%', 'PEN': '2.94%'}
Empresas con un rango por debajo del 0,6 %: ['ALM', 'AMCR', 'ASTS', 'FER', 'FERG', 'FLUT', 'HIMS', 'SW', 'VRT'] · con señales en el estudio: []


### 1.4 Las noticias de las sesiones previas a cada señal

Para cada señal se buscan los eventos de noticias de su empresa en las sesiones anteriores (5, y 3 como sensibilidad), sin la de la señal. La media de su sentimiento decide el grupo.

In [6]:
NOMBRE_DEL_GRUPO = {1: "positivas", 0: "neutrales", -1: "negativas"}


def grupo_por_noticias_previas(senales_estudio: pd.DataFrame, ventana: int) -> pd.Series:
    """Grupo de cada señal según la media del sentimiento de los eventos de sus `ventana` sesiones anteriores."""
    pares = senales_estudio[["ticker", "fecha", "sesion_numero"]].merge(
        eventos.assign(sesion_evento=eventos["sesion"].map(numero_de_sesion))[["ticker", "sesion_evento", "sentimiento_medio"]],
        on="ticker")
    en_ventana = pares[(pares["sesion_evento"] >= pares["sesion_numero"] - ventana) & (pares["sesion_evento"] < pares["sesion_numero"])]
    media = en_ventana.groupby(["ticker", "fecha"])["sentimiento_medio"].mean()
    grupo = np.sign(media).astype(int).map(NOMBRE_DEL_GRUPO)
    return (senales_estudio.set_index(["ticker", "fecha"]).index.map(grupo).to_series(index=senales_estudio.index)
            .fillna("sin noticias"))


estudio["grupo"] = grupo_por_noticias_previas(estudio, VENTANA)
estudio["grupo_3_sesiones"] = grupo_por_noticias_previas(estudio, VENTANA_SENSIBILIDAD)

ORDEN = ["positivas", "neutrales", "negativas", "sin noticias"]
for ventana, columna in [(VENTANA, "grupo"), (VENTANA_SENSIBILIDAD, "grupo_3_sesiones")]:
    tabla = pd.crosstab(estudio["senal"], estudio[columna])[ORDEN]
    print(f"Noticias en las {ventana} sesiones previas:")
    display(pd.concat([tabla, (100 * tabla.div(tabla.sum(axis=1), axis=0)).round(1).add_suffix(" (%)")], axis=1))

Noticias en las 5 sesiones previas:


grupo,positivas,neutrales,negativas,sin noticias,positivas (%),neutrales (%),negativas (%),sin noticias (%)
senal,,,,,,,,
compra,7925,7977,2343,14080,24.5,24.7,7.2,43.6
venta,4279,5475,2718,9204,19.7,25.3,12.5,42.5


Noticias en las 3 sesiones previas:


grupo_3_sesiones,positivas,neutrales,negativas,sin noticias,positivas (%),neutrales (%),negativas (%),sin noticias (%)
senal,,,,,,,,
compra,5616,6914,1622,18173,17.4,21.4,5.0,56.2
venta,2992,4656,1976,12052,13.8,21.5,9.1,55.6


**Conclusión:** de 215.928 primeros días de racha quedan **54.001 señales** en las 621 empresas con noticias (32.325 compras y 21.676 ventas, de enero de 2017 a junio de 2020). No hace falta quitar ninguna empresa: las 7 del notebook 05 se movían con normalidad en 2016–2020 (rango diario del 1,9 % al 4,0 %), y las 9 que entonces casi no se movían (varias eran sociedades instrumentales a unos 10 dólares) no tienen noticias, así que no entran.

Con la ventana de 5 sesiones, el **57 % de las señales tiene noticias previas**. Los grupos tienen tamaños suficientes para comparar; el más pequeño son las compras con noticias negativas (2.343). Ya se ve una relación: **las compras llegan más a menudo después de noticias positivas** (24,5 %, frente al 7,2 % de negativas), y las ventas tienen proporcionalmente más negativas (12,5 %). Es lógico: las señales técnicas saltan con los movimientos del precio, y esos movimientos coinciden con las noticias. Con 3 sesiones, el reparto es parecido, con menos señales con noticias (44 %).

## 2. ¿Le va distinto a la señal según las noticias previas?

**La medida:** la rentabilidad de la acción a **5 y 10 sesiones** desde el cierre de la sesión de la señal, **menos la de SPY** en las mismas sesiones (la rentabilidad anormal del notebook 08, acumulada). **En las ventas se cambia el signo:** una venta acierta si la acción baja frente al mercado. Así, en las dos señales, **un valor positivo es que la señal acierta**.

**La referencia, solo análisis técnico:** todas las señales, sin mirar las noticias. Es lo que da la señal por sí sola.

**Las comparaciones, fijadas antes de ver los resultados:**
1. **Positivas − negativas:** ¿cambia el resultado de la señal según el signo de las noticias previas?
2. **Positivas − sin noticias:** ¿mejora una señal confirmada por noticias positivas frente a una sin noticias?

Las dos con el intervalo de Welch. **Sensibilidad:** con la ventana de 3 sesiones y sin recortar las colas.

**Como en el 08,** las colas se recortan a los percentiles 1 y 99, ahora de las rentabilidades a 5 y a 10 sesiones de todas las empresas en el periodo del estudio. **Limitación:** las señales de un mismo periodo se solapan (dos señales de una empresa a pocos días comparten sesiones, y los días de pánico afectan a todas), así que el intervalo de Welch es algo optimista.

In [7]:
HORIZONTES = [5, 10]   # sesiones después de la señal

cierres = (pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet", columns=["fecha", "ticker", "close"])
           .pivot(index="fecha", columns="ticker", values="close").reindex(sesiones).rename_axis("fecha"))
spy = pd.read_parquet(CARPETA_CACHE / "precios_spy.parquet").set_index("fecha")["close"].reindex(sesiones)
signo = np.where(estudio["senal"] == "venta", -1, 1)   # una venta acierta si la acción baja
indice_de_las_senales = pd.MultiIndex.from_arrays([estudio["fecha"], estudio["ticker"]])

LIMITES = {}
for horizonte in HORIZONTES:
    # Para cada sesión y empresa: del cierre de esa sesión al de `horizonte` sesiones después, menos lo mismo en SPY
    anormal = cierres.shift(-horizonte) / cierres - 1 - (spy.shift(-horizonte) / spy - 1).to_numpy()[:, None]
    anormal = anormal.stack()
    en_el_estudio = anormal[anormal.index.get_level_values("fecha").to_series().between(estudio["fecha"].min(), estudio["fecha"].max()).to_numpy()]
    LIMITES[horizonte] = en_el_estudio.quantile([0.01, 0.99]).to_numpy()
    bruta = anormal.reindex(indice_de_las_senales).to_numpy()
    estudio[f"a_{horizonte}"] = signo * bruta
    estudio[f"a_{horizonte}_recortada"] = signo * np.clip(bruta, *LIMITES[horizonte])
    print(f"{horizonte} sesiones · límites del recorte: del {LIMITES[horizonte][0]:.2%} al {LIMITES[horizonte][1]:.2%}"
          f" · señales sin rentabilidad: {np.isnan(bruta).sum()}")

5 sesiones · límites del recorte: del -13.52% al 15.33% · señales sin rentabilidad: 0
10 sesiones · límites del recorte: del -18.07% al 21.10% · señales sin rentabilidad: 0


### 2.1 Resultado de cada grupo, frente a la referencia

Rentabilidad anormal media (en %, positiva = la señal acierta) con su intervalo del 95 %, para compras y ventas. Ventana de noticias de 5 sesiones y colas recortadas.

In [8]:
ORDEN_CON_REFERENCIA = ["todas (solo análisis técnico)"] + ORDEN


def resumen_por_grupo(columna: str, columna_grupo: str = "grupo") -> pd.DataFrame:
    """Media de `columna` (en %) con su intervalo del 95 % para todas las señales y para cada grupo, en compras y ventas."""
    filas = {}
    for senal in ["compra", "venta"]:
        de_la_senal = estudio[(estudio["senal"] == senal) & estudio[columna].notna()]
        for grupo in ORDEN_CON_REFERENCIA:
            valores = de_la_senal[columna] if grupo == ORDEN_CON_REFERENCIA[0] else de_la_senal.loc[de_la_senal[columna_grupo] == grupo, columna]
            r = media_con_intervalo(valores)
            filas[(senal, grupo)] = {"señales": r["n"], "media (%)": 100 * r["media"],
                                     "IC 95 % inferior": 100 * r["ic_inferior"], "IC 95 % superior": 100 * r["ic_superior"]}
    return pd.DataFrame(filas).T.astype({"señales": int}).round(3)


for horizonte in HORIZONTES:
    print(f"A {horizonte} sesiones:")
    display(resumen_por_grupo(f"a_{horizonte}_recortada"))

A 5 sesiones:


señales  media (%)  IC 95 % inferior  \
compra todas (solo análisis técnico)    32325     -0.152            -0.193   
       positivas                         7925     -0.141            -0.223   
       neutrales                         7977     -0.134            -0.215   
       negativas                         2343     -0.448            -0.633   
       sin noticias                     14080     -0.119            -0.178   
venta  todas (solo análisis técnico)    21676      0.016            -0.047   
       positivas                         4279      0.035            -0.098   
       neutrales                         5475      0.114            -0.006   
       negativas                         2718      0.083            -0.139   
       sin noticias                      9204     -0.071            -0.163   

                                      IC 95 % superior  
compra todas (solo análisis técnico)            -0.112  
       positivas                                -0.060  
       neutrales                                -0.053  
       negativas                                -0.263  
       sin noticias                             -0.060  
venta  todas (solo análisis técnico)             0.079  
       positivas                                 0.168  
       neutrales                                 0.235  
       negativas                                 0.306  
       sin noticias                              0.022

A 10 sesiones:


señales  media (%)  IC 95 % inferior  \
compra todas (solo análisis técnico)    32325     -0.201            -0.258   
       positivas                         7925     -0.143            -0.257   
       neutrales                         7977     -0.170            -0.282   
       negativas                         2343     -0.637            -0.875   
       sin noticias                     14080     -0.180            -0.263   
venta  todas (solo análisis técnico)    21676     -0.140            -0.225   
       positivas                         4279     -0.002            -0.181   
       neutrales                         5475     -0.050            -0.218   
       negativas                         2718     -0.279            -0.564   
       sin noticias                      9204     -0.217            -0.344   

                                      IC 95 % superior  
compra todas (solo análisis técnico)            -0.145  
       positivas                                -0.029  
       neutrales                                -0.058  
       negativas                                -0.399  
       sin noticias                             -0.096  
venta  todas (solo análisis técnico)            -0.055  
       positivas                                 0.176  
       neutrales                                 0.118  
       negativas                                 0.007  
       sin noticias                             -0.089

### 2.2 Las comparaciones

Diferencia de medias entre grupos (en puntos porcentuales), con su intervalo de Welch.

In [9]:
COMPARACIONES = {"positivas − negativas": ("positivas", "negativas"), "positivas − sin noticias": ("positivas", "sin noticias")}


def comparar_grupos(columna: str, columna_grupo: str = "grupo") -> pd.DataFrame:
    """Cada comparación, en compras y ventas: diferencia de medias de `columna` (en %), intervalo de Welch y p-valor."""
    filas = {}
    for senal in ["compra", "venta"]:
        de_la_senal = estudio[(estudio["senal"] == senal) & estudio[columna].notna()]
        for nombre, (grupo_a, grupo_b) in COMPARACIONES.items():
            w = diferencia_medias_welch(de_la_senal.loc[de_la_senal[columna_grupo] == grupo_a, columna],
                                        de_la_senal.loc[de_la_senal[columna_grupo] == grupo_b, columna])
            filas[(senal, nombre)] = {"diferencia (%)": round(100 * w["diferencia"], 3), "IC 95 % inferior": round(100 * w["ic_inferior"], 3),
                                      "IC 95 % superior": round(100 * w["ic_superior"], 3), "p-valor": round(w["p_valor"], 4),
                                      "¿concluyente?": "sí" if w["ic_inferior"] > 0 or w["ic_superior"] < 0 else "no (el 0 está dentro)"}
    return pd.DataFrame(filas).T


pd.concat({f"{horizonte} sesiones": comparar_grupos(f"a_{horizonte}_recortada") for horizonte in HORIZONTES})

diferencia (%) IC 95 % inferior  \
5 sesiones  compra positivas − negativas             0.307            0.105   
                   positivas − sin noticias         -0.022           -0.123   
            venta  positivas − negativas            -0.048           -0.308   
                   positivas − sin noticias          0.106           -0.057   
10 sesiones compra positivas − negativas             0.494             0.23   
                   positivas − sin noticias          0.036           -0.105   
            venta  positivas − negativas             0.276           -0.061   
                   positivas − sin noticias          0.214           -0.006   

                                            IC 95 % superior p-valor  \
5 sesiones  compra positivas − negativas               0.509  0.0029   
                   positivas − sin noticias            0.079  0.6676   
            venta  positivas − negativas               0.211  0.7154   
                   positivas − sin noticias            0.268  0.2019   
10 sesiones compra positivas − negativas               0.758  0.0002   
                   positivas − sin noticias            0.178  0.6135   
            venta  positivas − negativas               0.613  0.1082   
                   positivas − sin noticias            0.434  0.0563   

                                                     ¿concluyente?  
5 sesiones  compra positivas − negativas                        sí  
                   positivas − sin noticias  no (el 0 está dentro)  
            venta  positivas − negativas     no (el 0 está dentro)  
                   positivas − sin noticias  no (el 0 está dentro)  
10 sesiones compra positivas − negativas                        sí  
                   positivas − sin noticias  no (el 0 está dentro)  
            venta  positivas − negativas     no (el 0 está dentro)  
                   positivas − sin noticias  no (el 0 está dentro)

**Sensibilidad:** las mismas comparaciones con la ventana de noticias de 3 sesiones, y sin recortar las colas.

In [10]:
pd.concat({
    (f"{horizonte} sesiones", version): tabla
    for horizonte in HORIZONTES
    for version, tabla in [("noticias de 3 sesiones", comparar_grupos(f"a_{horizonte}_recortada", "grupo_3_sesiones")),
                           ("sin recortar", comparar_grupos(f"a_{horizonte}"))]
})

diferencia (%)  \
5 sesiones  noticias de 3 sesiones compra positivas − negativas             0.278   
                                          positivas − sin noticias          0.005   
                                   venta  positivas − negativas             0.071   
                                          positivas − sin noticias          0.075   
            sin recortar           compra positivas − negativas             0.345   
                                          positivas − sin noticias          0.002   
                                   venta  positivas − negativas            -0.036   
                                          positivas − sin noticias          0.087   
10 sesiones noticias de 3 sesiones compra positivas − negativas             0.308   
                                          positivas − sin noticias           0.06   
                                   venta  positivas − negativas              0.34   
                                          positivas − sin noticias          0.148   
            sin recortar           compra positivas − negativas             0.516   
                                          positivas − sin noticias          0.056   
                                   venta  positivas − negativas             0.222   
                                          positivas − sin noticias          0.219   

                                                                   IC 95 % inferior  \
5 sesiones  noticias de 3 sesiones compra positivas − negativas               0.028   
                                          positivas − sin noticias           -0.107   
                                   venta  positivas − negativas              -0.238   
                                          positivas − sin noticias           -0.102   
            sin recortar           compra positivas − negativas                0.12   
                                          positivas − sin noticias           -0.107   
                                   venta  positivas − negativas              -0.395   
                                          positivas − sin noticias           -0.109   
10 sesiones noticias de 3 sesiones compra positivas − negativas              -0.018   
                                          positivas − sin noticias           -0.096   
                                   venta  positivas − negativas              -0.055   
                                          positivas − sin noticias           -0.091   
            sin recortar           compra positivas − negativas               0.233   
                                          positivas − sin noticias           -0.097   
                                   venta  positivas − negativas              -0.192   
                                          positivas − sin noticias           -0.038   

                                                                   IC 95 % superior  \
5 sesiones  noticias de 3 sesiones compra positivas − negativas               0.528   
                                          positivas − sin noticias            0.116   
                                   venta  positivas − negativas                0.38   
                                          positivas − sin noticias            0.253   
            sin recortar           compra positivas − negativas                0.57   
                                          positivas − sin noticias            0.111   
                                   venta  positivas − negativas               0.323   
                                          positivas − sin noticias            0.284   
10 sesiones noticias de 3 sesiones compra positivas − negativas               0.634   
                                          positivas − sin noticias            0.216   
                                   venta  positivas − negativas               0.736   
                                          positivas − sin noticias            0.388   
          

En un gráfico: la rentabilidad anormal media de cada grupo y de todas las señales, con su intervalo del 95 % (ventana de 5 sesiones, colas recortadas).

In [11]:
medias = pd.concat({
    f"{horizonte} sesiones": resumen_por_grupo(f"a_{horizonte}_recortada") for horizonte in HORIZONTES
}, names=["horizonte", "senal", "grupo"]).reset_index()
medias["error (%)"] = medias["media (%)"] - medias["IC 95 % inferior"]
grafico = px.bar(medias, x="grupo", y="media (%)", error_y="error (%)", color="horizonte", barmode="group", facet_col="senal",
                 category_orders={"grupo": ORDEN_CON_REFERENCIA, "senal": ["compra", "venta"]},
                 labels={"media (%)": "rentabilidad anormal media (%, positiva = acierta)", "grupo": "", "senal": "señal"},
                 title="Resultado de las señales técnicas según las noticias de las 5 sesiones previas (IC 95 %)")
grafico.show(renderer="plotly_mimetype")

**Conclusión: las noticias no mejoran una señal de compra, pero las negativas la empeoran.**

| Compras (5 sesiones de noticias, colas recortadas) | A 5 sesiones | A 10 sesiones |
|---|---|---|
| Todas, solo análisis técnico | −0,15 % | −0,20 % |
| Con noticias positivas antes | −0,14 % | −0,14 % |
| Sin noticias | −0,12 % | −0,18 % |
| **Con noticias negativas antes** | **−0,45 %** | **−0,64 %** |
| **Positivas − negativas (IC 95 %)** | **+0,31** (0,10 – 0,51) | **+0,49** (0,23 – 0,76) |

- **Por sí solas, las señales técnicas no baten al mercado** en este periodo: las compras quedan por debajo de SPY y las ventas no aciertan.
- **Comprar después de noticias negativas sale claramente peor:** la diferencia entre positivas y negativas es concluyente a 5 y a 10 sesiones. Se mantiene sin recortar las colas (+0,35 y +0,52) y con la ventana de 3 sesiones a 5 sesiones (+0,28); a 10 sesiones con 3 sesiones de noticias queda justo en el límite (+0,31, de −0,02 a 0,63).
- **Las noticias positivas no aportan:** positivas − sin noticias no es concluyente en ningún caso.
- **En las ventas no hay ninguna diferencia concluyente.**

**Cómo leerlo:** las noticias sirven como aviso, no como confirmación: una compra técnica contra noticias malas suele salir peor. No es una estrategia: incluso el mejor grupo queda por debajo del mercado, no hay costes de operación, y el intervalo de Welch es algo optimista porque las señales se solapan en el tiempo.

### Resultados para la app

Las tablas finales de este notebook se guardan en `data/cache/`, para que la app las enseñe sin volver a calcular nada.

In [12]:
resultados_senales = pd.concat({f"{horizonte} sesiones": resumen_por_grupo(f"a_{horizonte}_recortada") for horizonte in HORIZONTES},
                               names=["horizonte", "senal", "grupo"]).reset_index()
resultados_comparaciones = pd.concat({
    (f"{horizonte} sesiones", version): tabla
    for horizonte in HORIZONTES
    for version, tabla in [("principal", comparar_grupos(f"a_{horizonte}_recortada")),
                           ("noticias de 3 sesiones", comparar_grupos(f"a_{horizonte}_recortada", "grupo_3_sesiones")),
                           ("sin recortar", comparar_grupos(f"a_{horizonte}"))]
}, names=["horizonte", "version", "senal", "comparacion"]).reset_index().infer_objects()

resultados_senales.to_parquet(CARPETA_CACHE / "resultados_09_senales.parquet", index=False)
resultados_comparaciones.to_parquet(CARPETA_CACHE / "resultados_09_comparaciones.parquet", index=False)
print(f"Guardados: resultados_09_senales.parquet ({len(resultados_senales)} filas) y resultados_09_comparaciones.parquet ({len(resultados_comparaciones)} filas)")

Guardados: resultados_09_senales.parquet (20 filas) y resultados_09_comparaciones.parquet (24 filas)


## Comprobaciones

In [13]:
# Señales: primer día de racha, solo empresas con noticias y fechas con la ventana completa, y grupos sin mirar al futuro
assert estudio["senal"].isin(["compra", "venta"]).all() and not estudio.duplicated(["ticker", "fecha"]).any()
anterior = senales.set_index(["ticker", "fecha"])["senal"].groupby(level="ticker").shift()
assert (anterior.reindex(estudio.set_index(["ticker", "fecha"]).index).to_numpy() != estudio["senal"].to_numpy()).all()
assert estudio["ticker"].isin(eventos["ticker"]).all()
assert (estudio["fecha"] >= primera_sesion_con_ventana).all() and (estudio["fecha"] <= ultima_sesion_con_noticias).all()
assert estudio["grupo"].isin(ORDEN).all() and estudio["grupo_3_sesiones"].isin(ORDEN).all()

# Una señal sin noticias en 5 sesiones tampoco las tiene en 3; y un ejemplo recalculado a mano
assert (estudio.loc[estudio["grupo"] == "sin noticias", "grupo_3_sesiones"] == "sin noticias").all()
ejemplo = estudio[estudio["grupo"] != "sin noticias"].sample(1, random_state=SEMILLA).iloc[0]
del_ejemplo = eventos[(eventos["ticker"] == ejemplo["ticker"])
                      & eventos["sesion"].isin(sesiones[ejemplo["sesion_numero"] - VENTANA : ejemplo["sesion_numero"]])]
assert NOMBRE_DEL_GRUPO[int(np.sign(del_ejemplo["sentimiento_medio"].mean()))] == ejemplo["grupo"]
assert (del_ejemplo["sesion"] < ejemplo["fecha"]).all()   # todas las noticias son anteriores a la señal
print("Señales y noticias previas: OK")

Señales y noticias previas: OK


In [14]:
# Rentabilidades: un ejemplo recalculado a mano, el signo cambiado en las ventas, el recorte a sus límites,
# y los grupos sin solaparse y sumando todas las señales
for senal in ["compra", "venta"]:
    ejemplo = estudio[(estudio["senal"] == senal) & estudio["a_5"].notna()].sample(1, random_state=SEMILLA).iloc[0]
    n = ejemplo["sesion_numero"]
    accion = cierres[ejemplo["ticker"]].iloc[n + 5] / cierres[ejemplo["ticker"]].iloc[n] - 1
    mercado = spy.iloc[n + 5] / spy.iloc[n] - 1
    assert np.isclose(ejemplo["a_5"], (1 if senal == "compra" else -1) * (accion - mercado))
for horizonte in HORIZONTES:
    recortada = estudio[f"a_{horizonte}_recortada"].dropna() * np.where(estudio.loc[estudio[f"a_{horizonte}_recortada"].notna(), "senal"] == "venta", -1, 1)
    assert recortada.between(*LIMITES[horizonte]).all()
    resumen = resumen_por_grupo(f"a_{horizonte}_recortada")
    for senal in ["compra", "venta"]:
        assert resumen.loc[(senal, ORDEN), "señales"].sum() == resumen.loc[(senal, ORDEN_CON_REFERENCIA[0]), "señales"]
print("Rentabilidades después de las señales: OK")

Rentabilidades después de las señales: OK


In [15]:
# Resultados para la app: lo guardado es lo mismo que las tablas de la sección 2
guardado = pd.read_parquet(CARPETA_CACHE / "resultados_09_senales.parquet").set_index(["horizonte", "senal", "grupo"])
for horizonte in HORIZONTES:
    assert np.allclose(guardado.loc[f"{horizonte} sesiones", "media (%)"], resumen_por_grupo(f"a_{horizonte}_recortada")["media (%)"])
comparaciones = pd.read_parquet(CARPETA_CACHE / "resultados_09_comparaciones.parquet").set_index(["horizonte", "version", "senal", "comparacion"])
assert np.allclose(comparaciones.loc[("5 sesiones", "principal"), "diferencia (%)"], comparar_grupos("a_5_recortada")["diferencia (%)"].astype(float))
print("Resultados para la app: OK")

Resultados para la app: OK


/var/folders/y3/4d4_8dbn2q1dj5tqgdbsg6hc0000gn/T/ipykernel_4036/1720618238.py:4: PerformanceWarning:

indexing past lexsort depth may impact performance.

/var/folders/y3/4d4_8dbn2q1dj5tqgdbsg6hc0000gn/T/ipykernel_4036/1720618238.py:4: PerformanceWarning:

indexing past lexsort depth may impact performance.

/var/folders/y3/4d4_8dbn2q1dj5tqgdbsg6hc0000gn/T/ipykernel_4036/1720618238.py:6: PerformanceWarning:

indexing past lexsort depth may impact performance.

